# Module 3: Random Effects Implementation

## Learning Objectives

By completing this notebook, you will:
1. Understand the random effects model and its assumptions
2. Implement GLS estimation for random effects from scratch
3. Apply RE using linearmodels (Python) and plm (R)
4. Compute variance components (within vs between)
5. Recognize when RE assumptions are violated

## Prerequisites

- Module 2: Fixed Effects (completed)
- Understanding of GLS
- Variance components decomposition

## Estimated Time

75-90 minutes

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seed
np.random.seed(42)

print("✅ Setup complete")

## 1. The Random Effects Model

### Model Specification

$$y_{it} = \alpha + X_{it}\beta + \mu_i + \epsilon_{it}$$

where:
- $\alpha$ is a **common intercept** (not entity-specific)
- $\mu_i \sim N(0, \sigma^2_\mu)$ is the **random entity effect**
- $\epsilon_{it} \sim N(0, \sigma^2_\epsilon)$ is the **idiosyncratic error**

### Key Assumption

**Exogeneity of entity effects:**

$$E[\mu_i | X_{i1}, ..., X_{iT}] = 0$$

This means: **Entity effects UNCORRELATED with predictors**.

### Composite Error

Rewrite as:

$$y_{it} = \alpha + X_{it}\beta + v_{it}$$

where $v_{it} = \mu_i + \epsilon_{it}$ is the **composite error**.

### Why Not OLS?

The composite error creates **serial correlation**:

$$Cov(v_{it}, v_{is}) = \sigma^2_\mu \text{ for } t \neq s$$

OLS is **inefficient** (but consistent if assumption holds). **GLS** is more efficient.

## 2. Generate Data Satisfying RE Assumptions

Create data where entity effects are **uncorrelated** with X.

In [ ]:
# Generate panel data for RE
n_entities = 100
n_periods = 10

# Random entity effects (INDEPENDENT of X)
mu_i = np.random.randn(n_entities) * 10

# Generate X (INDEPENDENT of mu_i)
X = []
for i in range(n_entities):
    # X varies randomly, NOT correlated with mu_i
    X_i = []
    for t in range(n_periods):
        X_i.append(20 + np.random.randn() * 5)
    X.extend(X_i)

X = np.array(X)

# Generate y
mu_repeated = np.repeat(mu_i, n_periods)
epsilon = np.random.randn(n_entities * n_periods) * 5

# True model: y = 50 + 1.5*X + mu_i + epsilon
alpha_true = 50
beta_true = 1.5
y = alpha_true + beta_true * X + mu_repeated + epsilon

# True variance components
sigma_mu_true = 10
sigma_epsilon_true = 5

# Create DataFrame
entities = np.repeat(np.arange(1, n_entities + 1), n_periods)
periods = np.tile(np.arange(1, n_periods + 1), n_entities)

df = pd.DataFrame({
    'entity_id': entities,
    'time': periods,
    'y': y,
    'x': X
})

print("Generated Random Effects Data")
print(f"  Entities: {n_entities}")
print(f"  Periods: {n_periods}")
print(f"  True parameters:")
print(f"    Alpha (intercept): {alpha_true}")
print(f"    Beta (slope): {beta_true}")
print(f"    Sigma_mu (entity variance): {sigma_mu_true}")
print(f"    Sigma_epsilon (error variance): {sigma_epsilon_true}")
print(f"\nKey: mu_i is UNCORRELATED with X (RE assumption satisfied)")
print(f"\nFirst 15 rows:")
print(df.head(15))

### Verify RE Assumption: Cov(mu_i, X) = 0

In [ ]:
# Compute entity-level means
entity_means = df.groupby('entity_id').agg({
    'x': 'mean'
}).reset_index()

entity_means['true_mu'] = mu_i

# Correlation between entity effect and X
corr = entity_means[['x', 'true_mu']].corr().iloc[0, 1]

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(entity_means['x'], entity_means['true_mu'], alpha=0.5, s=50)
ax.axhline(y=0, color='red', linestyle='--', linewidth=1)

ax.set_xlabel('Mean of X (by entity)', fontsize=12)
ax.set_ylabel('True Random Effect (mu_i)', fontsize=12)
ax.set_title(f'RE Assumption Check: Cov(mu_i, X) = {corr:.4f}', fontsize=14)
ax.grid(True, alpha=0.3)

# Add text
ax.text(0.05, 0.95, f'Correlation: {corr:.4f}',
        transform=ax.transAxes, fontsize=12,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

if abs(corr) < 0.15:
    print("✅ RE assumption satisfied: mu_i uncorrelated with X")
else:
    print("⚠️  RE assumption violated: mu_i correlated with X")

## 3. Pooled OLS: Consistent but Inefficient

Under RE assumptions, pooled OLS is **consistent** but **inefficient**.

Standard errors are wrong due to serial correlation in composite error.

In [ ]:
# Pooled OLS
X_pooled = sm.add_constant(df['x'])
y_pooled = df['y']

model_pooled = sm.OLS(y_pooled, X_pooled)
results_pooled = model_pooled.fit()

print("POOLED OLS Results")
print("=" * 70)
print(results_pooled.summary().tables[1])
print("\n" + "=" * 70)
print(f"True parameters:")
print(f"  Alpha: {alpha_true:.4f}")
print(f"  Beta:  {beta_true:.4f}")
print(f"\nPooled OLS estimates:")
print(f"  Alpha: {results_pooled.params['const']:.4f}")
print(f"  Beta:  {results_pooled.params['x']:.4f}")
print("\n✅ Pooled OLS is CONSISTENT (estimates are close to truth)")
print("⚠️  But standard errors are WRONG (ignore serial correlation)")

## 4. Random Effects: GLS Estimation

### GLS Transformation

**Step 1:** Estimate variance components

From OLS residuals:
- Within variance: $\hat{\sigma}^2_\epsilon$
- Between variance: $\hat{\sigma}^2_\mu + \hat{\sigma}^2_\epsilon / T$

**Step 2:** Compute transformation parameter

$$\theta = 1 - \sqrt{\frac{\sigma^2_\epsilon}{\sigma^2_\epsilon + T\sigma^2_\mu}}$$

**Step 3:** Transform data

$$\tilde{y}_{it} = y_{it} - \theta\bar{y}_i$$
$$\tilde{X}_{it} = X_{it} - \theta\bar{X}_i$$

**Step 4:** OLS on transformed data

In [ ]:
def random_effects_gls(df, entity_col, y_col, X_cols):
    """
    Estimate random effects using GLS.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    y_col : str
    X_cols : list
    
    Returns
    -------
    dict
        Estimation results
    """
    # Step 1: Estimate variance components using pooled OLS
    X_pool = sm.add_constant(df[X_cols])
    y_pool = df[y_col]
    pooled = sm.OLS(y_pool, X_pool).fit()
    
    # Within residuals
    df_temp = df.copy()
    df_temp['resid_pooled'] = pooled.resid
    
    # Entity means
    entity_means_y = df.groupby(entity_col)[y_col].mean()
    entity_means_X = df.groupby(entity_col)[X_cols].mean()
    
    # Between residuals
    if len(X_cols) == 1:
        X_means_array = entity_means_X.values.reshape(-1, 1)
    else:
        X_means_array = entity_means_X.values
    
    X_between = sm.add_constant(X_means_array)
    y_between = entity_means_y.values
    
    between = sm.OLS(y_between, X_between).fit()
    
    # Variance estimates
    T = df.groupby(entity_col).size().mean()
    
    # Within variance (from pooled OLS residuals)
    sigma2_epsilon = pooled.mse_resid
    
    # Between variance
    sigma2_between = between.mse_resid
    sigma2_mu = max(0, sigma2_between - sigma2_epsilon / T)
    
    # Step 2: Compute theta
    theta = 1 - np.sqrt(sigma2_epsilon / (sigma2_epsilon + T * sigma2_mu))
    
    # Step 3: Transform data
    entity_means_y_series = df.groupby(entity_col)[y_col].transform('mean')
    entity_means_X_series = df.groupby(entity_col)[X_cols].transform('mean')
    
    y_transformed = df[y_col] - theta * entity_means_y_series
    X_transformed = df[X_cols] - theta * entity_means_X_series
    
    # Also transform intercept
    intercept_transformed = np.ones(len(df)) * (1 - theta)
    X_transformed_with_const = pd.concat([
        pd.DataFrame({'const': intercept_transformed}),
        X_transformed
    ], axis=1)
    
    # Step 4: OLS on transformed data
    model_gls = sm.OLS(y_transformed, X_transformed_with_const)
    results_gls = model_gls.fit()
    
    return {
        'params': results_gls.params,
        'se': results_gls.bse,
        't_stats': results_gls.tvalues,
        'p_values': results_gls.pvalues,
        'r_squared': results_gls.rsquared,
        'sigma2_mu': sigma2_mu,
        'sigma2_epsilon': sigma2_epsilon,
        'theta': theta,
        'results': results_gls
    }

# Estimate RE
re_results = random_effects_gls(df, 'entity_id', 'y', ['x'])

print("RANDOM EFFECTS (GLS) Results")
print("=" * 70)
print(f"{'Variable':<15} {'Coef':>10} {'Std Err':>10} {'t':>8} {'P>|t|':>10}")
print("=" * 70)
for var in ['const', 'x']:
    print(f"{var:<15} {re_results['params'][var]:>10.4f} {re_results['se'][var]:>10.4f} "
          f"{re_results['t_stats'][var]:>8.3f} {re_results['p_values'][var]:>10.4f}")

print("\n" + "=" * 70)
print("Variance Components:")
print(f"  Sigma^2_mu (entity):       {re_results['sigma2_mu']:>10.4f} (true: {sigma_mu_true**2:.4f})")
print(f"  Sigma^2_epsilon (error):   {re_results['sigma2_epsilon']:>10.4f} (true: {sigma_epsilon_true**2:.4f})")
print(f"  Theta (transformation):    {re_results['theta']:>10.4f}")

print("\n" + "=" * 70)
print("Comparison to True Values:")
print(f"  True alpha: {alpha_true:.4f}  |  Estimated: {re_results['params']['const']:.4f}")
print(f"  True beta:  {beta_true:.4f}  |  Estimated: {re_results['params']['x']:.4f}")

print("\n✅ Random effects GLS is CONSISTENT and EFFICIENT")

### Understanding Theta

The parameter $\theta$ controls the transformation:

- **$\theta = 0$:** No transformation (pooled OLS)
  - Happens when $\sigma^2_\mu = 0$ (no entity effects)
  
- **$\theta = 1$:** Full demeaning (fixed effects)
  - Happens when $\sigma^2_\mu \to \infty$ (large entity effects)
  
- **$0 < \theta < 1$:** Partial demeaning (random effects)
  - Optimal weighted average of within and between variation

In [ ]:
# Visualize transformation
print(f"Transformation Parameter: theta = {re_results['theta']:.4f}")
print("\nInterpretation:")
print(f"  RE subtracts {re_results['theta']:.2%} of entity mean from each observation")
print(f"  FE would subtract 100% (theta = 1)")
print(f"  Pooled OLS subtracts 0% (theta = 0)")
print("\nRE is a weighted average between within and between variation")

# Compute weights
theta = re_results['theta']
within_weight = theta
between_weight = 1 - theta

print(f"\nVariation weights:")
print(f"  Within variation:  {within_weight:.2%}")
print(f"  Between variation: {between_weight:.2%}")

## 5. Random Effects Using linearmodels

In [ ]:
# Set panel index
df_panel = df.set_index(['entity_id', 'time'])

# Random Effects model
model_re = RandomEffects(
    dependent=df_panel['y'],
    exog=df_panel[['x']]
)

results_re = model_re.fit()

print("RANDOM EFFECTS (linearmodels) Results")
print("=" * 70)
print(results_re)

# Compare implementations
print("\n" + "=" * 70)
print("Comparison: Manual GLS vs linearmodels")
print("=" * 70)
print(f"{'Parameter':<15} {'Manual GLS':>15} {'linearmodels':>15} {'Difference':>15}")
print("=" * 70)

# Intercept
diff_alpha = abs(re_results['params']['const'] - results_re.params['const'])
print(f"{'Intercept':<15} {re_results['params']['const']:>15.6f} {results_re.params['const']:>15.6f} {diff_alpha:>15.2e}")

# Slope
diff_beta = abs(re_results['params']['x'] - results_re.params['x'])
print(f"{'Slope (x)':<15} {re_results['params']['x']:>15.6f} {results_re.params['x']:>15.6f} {diff_beta:>15.2e}")

print("\n✅ Implementations match!")

## 6. Compare: Pooled OLS vs FE vs RE

Let's compare all three estimators on the same data.

In [ ]:
# Already have pooled OLS
# Estimate FE
model_fe = PanelOLS(
    dependent=df_panel['y'],
    exog=df_panel[['x']],
    entity_effects=True
)
results_fe = model_fe.fit(cov_type='clustered', cluster_entity=True)

# Compare
print("Comparison: Pooled OLS vs FE vs RE")
print("=" * 70)
print(f"{'Estimator':<20} {'Beta':>12} {'Std Err':>12} {'95% CI Width':>15}")
print("=" * 70)

# True
print(f"{'True value':<20} {beta_true:>12.4f} {'':>12} {'':>15}")

# Pooled OLS
beta_pool = results_pooled.params['x']
se_pool = results_pooled.bse['x']
ci_width_pool = 1.96 * 2 * se_pool
print(f"{'Pooled OLS':<20} {beta_pool:>12.4f} {se_pool:>12.4f} {ci_width_pool:>15.4f}")

# Fixed Effects
beta_fe = results_fe.params['x']
se_fe = results_fe.std_errors['x']
ci_width_fe = 1.96 * 2 * se_fe
print(f"{'Fixed Effects':<20} {beta_fe:>12.4f} {se_fe:>12.4f} {ci_width_fe:>15.4f}")

# Random Effects
beta_re = results_re.params['x']
se_re = results_re.std_errors['x']
ci_width_re = 1.96 * 2 * se_re
print(f"{'Random Effects':<20} {beta_re:>12.4f} {se_re:>12.4f} {ci_width_re:>15.4f}")

print("\n" + "=" * 70)
print("Key Observations:")
print(f"  1. All three are consistent (close to true value)")
print(f"  2. RE has smallest standard error (most efficient)")
print(f"  3. FE has largest SE (discards between variation)")
print(f"  4. Pooled OLS SE is wrong (ignores serial correlation)")

# Efficiency gain
efficiency_gain = (se_fe - se_re) / se_fe * 100
print(f"\n  RE is {efficiency_gain:.1f}% more efficient than FE")

### Visualize Efficiency Comparison

In [ ]:
# Create confidence intervals
estimators = ['Pooled OLS', 'Fixed Effects', 'Random Effects']
betas = [beta_pool, beta_fe, beta_re]
ses = [se_pool, se_fe, se_re]

ci_lower = [b - 1.96*se for b, se in zip(betas, ses)]
ci_upper = [b + 1.96*se for b, se in zip(betas, ses)]

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

y_pos = np.arange(len(estimators))

# Confidence intervals
for i, (lower, upper, beta) in enumerate(zip(ci_lower, ci_upper, betas)):
    ax.plot([lower, upper], [i, i], 'o-', linewidth=2, markersize=8)
    ax.plot([beta], [i], 'ro', markersize=10)

# True value
ax.axvline(x=beta_true, color='green', linestyle='--', linewidth=2,
           label=f'True value ({beta_true})', alpha=0.7)

ax.set_yticks(y_pos)
ax.set_yticklabels(estimators)
ax.set_xlabel('Coefficient Value', fontsize=12)
ax.set_title('95% Confidence Intervals: Estimator Comparison', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("Note: Random Effects has narrowest CI (most precise estimates)")

---

## Exercises

### Exercise 1: Decompose Total Variance

**Task:** Decompose the total variance of $y$ into three components:
1. Explained by X
2. Entity effects (mu_i)
3. Idiosyncratic error (epsilon)

**Expected output:** Variance decomposition showing proportion of each

**Hints:**
- Use estimated variance components
- Total var = var(X*beta) + var(mu_i) + var(epsilon)

In [ ]:
# YOUR CODE HERE
def variance_decomposition(df, entity_col, y_col, X_cols, beta, 
                           sigma2_mu, sigma2_epsilon):
    """
    Decompose variance of y.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    y_col : str
    X_cols : list
    beta : float
        Estimated slope
    sigma2_mu : float
        Entity variance
    sigma2_epsilon : float
        Error variance
    
    Returns
    -------
    dict
        Variance decomposition
    """
    # TODO: Implement variance decomposition
    pass

In [ ]:
# SOLUTION (hidden)
def variance_decomposition_solution(df, entity_col, y_col, X_cols, beta, 
                                     sigma2_mu, sigma2_epsilon):
    """Decompose variance of y."""
    # Total variance
    var_total = df[y_col].var()
    
    # Variance explained by X
    X = df[X_cols[0]].values
    var_explained = np.var(beta * X)
    
    # Variance from entity effects
    var_entity = sigma2_mu
    
    # Variance from idiosyncratic error
    var_error = sigma2_epsilon
    
    # Proportions
    total_components = var_explained + var_entity + var_error
    prop_explained = var_explained / total_components
    prop_entity = var_entity / total_components
    prop_error = var_error / total_components
    
    return {
        'var_total': var_total,
        'var_explained': var_explained,
        'var_entity': var_entity,
        'var_error': var_error,
        'prop_explained': prop_explained,
        'prop_entity': prop_entity,
        'prop_error': prop_error
    }

In [ ]:
# Test your function
def test_exercise_1():
    """Test variance decomposition."""
    var_decomp = variance_decomposition(
        df, 'entity_id', 'y', ['x'],
        beta=re_results['params']['x'],
        sigma2_mu=re_results['sigma2_mu'],
        sigma2_epsilon=re_results['sigma2_epsilon']
    )
    
    assert var_decomp is not None, "Function returns None - did you implement it?"
    assert 'var_total' in var_decomp, "Missing var_total"
    
    # Proportions should sum to ~1
    prop_sum = (var_decomp['prop_explained'] + 
                var_decomp['prop_entity'] + 
                var_decomp['prop_error'])
    
    assert np.isclose(prop_sum, 1.0, atol=0.01), "Proportions should sum to 1"
    
    print("✅ Correct! Variance decomposition:")
    print("\nVariance Components:")
    print("=" * 50)
    print(f"Total variance:          {var_decomp['var_total']:.2f}")
    print(f"\nDecomposition:")
    print(f"  Explained by X:        {var_decomp['var_explained']:.2f} ({var_decomp['prop_explained']*100:.1f}%)")
    print(f"  Entity effects:        {var_decomp['var_entity']:.2f} ({var_decomp['prop_entity']*100:.1f}%)")
    print(f"  Idiosyncratic error:   {var_decomp['var_error']:.2f} ({var_decomp['prop_error']*100:.1f}%)")
    
    # Visualization
    fig, ax = plt.subplots(figsize=(8, 6))
    
    components = ['Explained\nby X', 'Entity\nEffects', 'Error']
    values = [var_decomp['prop_explained'], 
              var_decomp['prop_entity'], 
              var_decomp['prop_error']]
    
    ax.bar(components, values, alpha=0.7, edgecolor='black')
    ax.set_ylabel('Proportion of Variance', fontsize=12)
    ax.set_title('Variance Decomposition', fontsize=14)
    ax.set_ylim([0, 1])
    ax.grid(True, alpha=0.3, axis='y')
    
    for i, v in enumerate(values):
        ax.text(i, v + 0.02, f'{v*100:.1f}%', 
                ha='center', va='bottom', fontsize=11)
    
    plt.tight_layout()
    plt.show()
    
    return True

# Uncomment to test
# test_exercise_1()

### Exercise 2: Simulate RE vs FE Efficiency

**Task:** Run Monte Carlo simulation to compare RE and FE efficiency.

**Procedure:**
1. Generate 100 datasets under RE assumptions
2. Estimate beta with both RE and FE
3. Compute standard deviation of estimates (empirical SE)
4. Compare efficiency

**Expected:** RE should have lower empirical SE than FE

In [ ]:
# YOUR CODE HERE
def compare_efficiency_simulation(n_sims=100, n_entities=50, n_periods=8):
    """
    Monte Carlo comparison of RE vs FE efficiency.
    
    Parameters
    ----------
    n_sims : int
    n_entities : int
    n_periods : int
    
    Returns
    -------
    dict
        Empirical SEs for RE and FE
    """
    # TODO: Implement simulation
    pass

In [ ]:
# SOLUTION (hidden)
def compare_efficiency_simulation_solution(n_sims=100, n_entities=50, n_periods=8):
    """Monte Carlo comparison of RE vs FE efficiency."""
    beta_re_estimates = []
    beta_fe_estimates = []
    
    true_beta = 1.5
    
    for sim in range(n_sims):
        # Generate data
        mu_i = np.random.randn(n_entities) * 10
        
        X = []
        for i in range(n_entities):
            for t in range(n_periods):
                X.append(20 + np.random.randn() * 5)
        X = np.array(X)
        
        mu_repeated = np.repeat(mu_i, n_periods)
        epsilon = np.random.randn(n_entities * n_periods) * 5
        y = 50 + true_beta * X + mu_repeated + epsilon
        
        # Create DataFrame
        entities = np.repeat(np.arange(1, n_entities + 1), n_periods)
        periods = np.tile(np.arange(1, n_periods + 1), n_entities)
        df_sim = pd.DataFrame({
            'entity_id': entities,
            'time': periods,
            'y': y,
            'x': X
        }).set_index(['entity_id', 'time'])
        
        # Estimate RE
        try:
            model_re = RandomEffects(df_sim['y'], df_sim[['x']])
            results_re = model_re.fit()
            beta_re_estimates.append(results_re.params['x'])
        except:
            pass
        
        # Estimate FE
        try:
            model_fe = PanelOLS(df_sim['y'], df_sim[['x']], entity_effects=True)
            results_fe = model_fe.fit()
            beta_fe_estimates.append(results_fe.params['x'])
        except:
            pass
    
    # Empirical SEs
    empirical_se_re = np.std(beta_re_estimates)
    empirical_se_fe = np.std(beta_fe_estimates)
    
    return {
        'empirical_se_re': empirical_se_re,
        'empirical_se_fe': empirical_se_fe,
        'estimates_re': beta_re_estimates,
        'estimates_fe': beta_fe_estimates
    }

In [ ]:
# Test your function (WARNING: Takes ~30 seconds)
def test_exercise_2():
    """Test efficiency simulation."""
    print("Running simulation (this may take 30 seconds)...")
    
    sim_results = compare_efficiency_simulation(
        n_sims=50,  # Reduced for speed
        n_entities=50,
        n_periods=8
    )
    
    assert sim_results is not None, "Function returns None - did you implement it?"
    assert 'empirical_se_re' in sim_results, "Missing empirical_se_re"
    assert 'empirical_se_fe' in sim_results, "Missing empirical_se_fe"
    
    # RE should be more efficient
    efficiency_gain = (sim_results['empirical_se_fe'] - 
                       sim_results['empirical_se_re']) / sim_results['empirical_se_fe'] * 100
    
    print("\n✅ Correct! Simulation complete.")
    print("\nEmpirical Standard Errors (from simulation):")
    print("=" * 50)
    print(f"  Random Effects:  {sim_results['empirical_se_re']:.4f}")
    print(f"  Fixed Effects:   {sim_results['empirical_se_fe']:.4f}")
    print(f"\n  Efficiency gain: {efficiency_gain:.1f}%")
    
    # Plot distributions
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.hist(sim_results['estimates_re'], bins=20, alpha=0.5, 
            label=f'RE (SE={sim_results["empirical_se_re"]:.3f})', 
            color='blue', edgecolor='black')
    ax.hist(sim_results['estimates_fe'], bins=20, alpha=0.5, 
            label=f'FE (SE={sim_results["empirical_se_fe"]:.3f})', 
            color='red', edgecolor='black')
    
    ax.axvline(x=1.5, color='green', linestyle='--', linewidth=2, 
               label='True value (1.5)')
    
    ax.set_xlabel('Estimated Beta', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title('Distribution of Estimates: RE vs FE', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("\nNote: RE distribution is tighter (more efficient)")
    
    return True

# Uncomment to test (takes ~30 seconds)
# test_exercise_2()

---

## Summary

### Key Takeaways

1. **RE Model:** Treats entity effects as random draws from distribution

2. **Key Assumption:** Entity effects UNCORRELATED with predictors

3. **GLS Estimation:** Partial demeaning (weighted average of within and between)

4. **Efficiency:** RE is more efficient than FE when assumptions hold

5. **Theta Parameter:** Controls transformation (0 = pooled, 1 = FE, 0<theta<1 = RE)

### When to Use Random Effects

✅ **Use RE when:**
- Entity effects plausibly uncorrelated with predictors
- Want to estimate effects of time-invariant variables
- Entities are random sample from population
- Efficiency gain matters (small T)

❌ **Do NOT use RE when:**
- Entity effects correlated with predictors (use FE)
- Hausman test rejects (next notebook)
- Within-entity effects are the research question

### What's Next

In the next notebook, we'll:
- Compare RE vs FE directly
- Apply the Hausman test
- Understand trade-offs
- Practical guidance on model choice

---

**Next:** `02_re_vs_fe.ipynb` - Random Effects vs Fixed Effects